# 03 — Manifest → priors → scale-conditioned mass regressor (GPU job 2)

The novel model: mass regression conditioned on measured metric geometry (docs/MODELS.md §4). Requires **notebook 01** (Nutrition5k on Drive).

Steps: (1) extract the geometry manifest from the RGB-D captures — the same quantities the phone measures; (2) fit the κ/φ/h̄ shape priors used by the pure-geometry pipeline; (3) train the regressor.

- **Runtime:** manifest extraction ~30–60 min (CPU-bound); training ~1–2 h on H100 at batch 128.
- **Benchmarks to beat:** 26.1% calorie MAPE (RGB) / 16.5% (RGB+depth).
- All artifacts and caches persist to Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/physics-powered-portion-estimation'

# Framework caches stay LOCAL (fast, and mmap over Drive's FUSE mount is
# unreliable); the (small) pretrained backbone re-downloads in seconds.
# The DATASET and all OUTPUTS below live on Drive — that's what's worth keeping.
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['TORCH_HOME'] = '/content/torch-cache'

DATA = f'{DRIVE_ROOT}/data/nutrition5k'
OUT = f'{DRIVE_ROOT}/out'
CKPT = f'{DRIVE_ROOT}/checkpoints/mass-regressor.pt'
!mkdir -p "{OUT}"
!nvidia-smi | head -12

Mounted at /content/drive
/bin/bash: line 1: nvidia-smi: command not found


In [2]:
# Clone the repo. Private repo → add your GitHub token as a Colab secret named
# GH_TOKEN (🔑 left sidebar). Public repo → no token needed. Fails LOUD.
import os, subprocess
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN') or ''
except Exception:
    pass
repo = 'github.com/Hakeem-Hannoon/physics-powered-portion-estimation.git'
url = f'https://x-access-token:{token}@{repo}' if token else f'https://{repo}'
subprocess.run(['rm', '-rf', '/content/ppe'])
r = subprocess.run(['git', 'clone', '--depth', '1', url, '/content/ppe'],
                   capture_output=True, text=True)
assert os.path.isdir('/content/ppe/model'), (
    (r.stderr.replace(token, '***') if token else r.stderr) +
    '\nClone failed — add a GH_TOKEN Colab secret (🔑) or make the repo public.')
print('repo ready')

%pip -q install timm pandas

repo ready
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 74.0 MB/s eta 0:00:00


In [ ]:
%cd /content/ppe
# 1. Geometry manifest (area, height, volume per dish + mass/kcal labels).
!python model/data/prepare_nutrition5k.py --root "{DATA}" --out "{OUT}/n5k-manifest.csv"
!head -3 "{OUT}/n5k-manifest.csv"

/content/ppe


In [ ]:
# 2. Shape priors — the printed global κ replaces DEFAULT_KAPPA in
#    packages/pipeline/src/estimate.ts and feeds nutrition/'s shape_priors.
!python model/priors/fit_priors.py --manifest "{OUT}/n5k-manifest.csv" --out "{OUT}/priors.json"
!cat "{OUT}/priors.json"

In [ ]:
# 3. Train. Per-epoch mass/kcal MAPE prints; best checkpoint saves to Drive.
!python model/train/mass_regressor_nutrition5k.py \
  --manifest "{OUT}/n5k-manifest.csv" \
  --epochs 50 --batch-size 128 \
  --output "{CKPT}"
print('README row template:')
print('| Nutrition5k test split | calorie MAPE | 26.1% RGB / 16.5% depth | **<best kcal MAPE from above>** |')